# PCB-MC Notebook 10 (v2): Fold-Aware Two-Stage Pipeline

**What changed from v1:**
Each fold now uses its own patch classifier checkpoint trained on that fold's
training boards only. This matches the same board-identity-aware protocol
used for YOLO training, making results directly comparable to Table III.

**Pipeline per fold:**
- Stage 1: YOLOv11 `full_dataset` fold_i checkpoint (conf=0.05)
- Stage 2: ResNet-18 patch classifier fold_i checkpoint (from Notebook 9 v2)
- Evaluation: `missing_only/kfold_data/fold_i/valid/images`


## [0] Setup

In [1]:
%%capture
!pip install ultralytics scikit-learn

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## [1] Imports

In [3]:
import os, json, random, warnings
from pathlib import Path
from typing import List, Tuple

import cv2, numpy as np
from PIL import Image
import torch, torch.nn as nn
import torchvision.transforms as T
from torchvision import models
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from tqdm import tqdm
from ultralytics import YOLO

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
device: cuda


## [2] Paths & Configuration

In [4]:
DATA_ROOT   = '/content/drive/MyDrive/PCB_MC/Data'
YOLO_ROOT   = '/content/drive/MyDrive/PCB_MC/Results/YOLOV11'
CLF_ROOT    = '/content/drive/MyDrive/PCB_MC/Results/pcbmc_patch_clf_v3/checkpoints'
OUT_DIR     = '/content/drive/MyDrive/PCB_MC/Results/pcbmc_twostage_v3'
os.makedirs(OUT_DIR, exist_ok=True)

IMG_EXTS    = ('.jpg','.jpeg','.png','.bmp','.webp')
FOLDS       = 5
STAGE1_CONF = 0.05
STAGE1_IOU  = 0.45
STAGE1_IMGSZ= 1024
CLF_THR     = 0.30
CONTEXT_FACTOR = 1.4
CROP_SIZE   = 128
NMS_IOU     = 0.30
SEED        = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Config OK')


Config OK


## [3] Helpers

In [5]:
def list_images(folder):
    if not os.path.exists(folder): return []
    return sorted([os.path.join(folder,f) for f in os.listdir(folder)
                   if f.lower().endswith(IMG_EXTS)])

def read_yolo_labels(lp):
    if not os.path.exists(lp): return []
    rows=[]
    with open(lp) as f:
        for line in f:
            p=line.strip().split()
            if len(p)<5: continue
            rows.append((int(float(p[0])),*map(float,p[1:5])))
    return rows

def label_path(img_path):
    return img_path.replace('/images/','/labels/').rsplit('.',1)[0]+'.txt'

def yolo_to_xyxy(xc,yc,w,h,W,H):
    x1=int(max(0,(xc-w/2)*W)); y1=int(max(0,(yc-h/2)*H))
    x2=int(min(W-1,(xc+w/2)*W)); y2=int(min(H-1,(yc+h/2)*H))
    if x2<=x1: x2=min(W-1,x1+1)
    if y2<=y1: y2=min(H-1,y1+1)
    return x1,y1,x2,y2

def expand_bbox(x1,y1,x2,y2,f,W,H):
    cx,cy=(x1+x2)/2,(y1+y2)/2
    hw=(x2-x1)/2*f; hh=(y2-y1)/2*f
    return int(max(0,cx-hw)),int(max(0,cy-hh)),\
           int(min(W-1,cx+hw)),int(min(H-1,cy+hh))

def iou_xyxy(a,b):
    iw=max(0,min(a[2],b[2])-max(a[0],b[0]))
    ih=max(0,min(a[3],b[3])-max(a[1],b[1]))
    inter=iw*ih
    ua=(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter
    return float(inter/max(ua,1e-9))

def nms(boxes,iou_thr=NMS_IOU):
    keep=[]
    for b in sorted(boxes,key=lambda x:x[4],reverse=True):
        if all(iou_xyxy(b[:4],k[:4])<iou_thr for k in keep):
            keep.append(b)
    return keep

def compute_ap_11pt(scored_tp, total_gt):
    if total_gt==0: return 0.0
    scored_tp=sorted(scored_tp,key=lambda x:x[0],reverse=True)
    tp=fp=0; precs,recs=[],[]
    for s,is_tp in scored_tp:
        tp+=int(is_tp); fp+=int(1-is_tp)
        precs.append(tp/max(1,tp+fp)); recs.append(tp/total_gt)
    return float(sum(max([p for p,r in zip(precs,recs) if r>=t]+[0.])
                     for t in np.linspace(0,1,11))/11.)

def fold_val_imgs(subset, fold_idx):
    d = os.path.join(DATA_ROOT, subset, 'kfold_data',
                     f'fold_{fold_idx}', 'valid', 'images')
    return list_images(d)

print('Helpers OK')


Helpers OK


## [4] Load Stage-2 Classifier for a Fold

In [6]:
EVAL_TF = T.Compose([
    T.Resize((CROP_SIZE,CROP_SIZE)),
    T.ToTensor(),
    T.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

def build_classifier():
    m=models.resnet18(weights=None)
    m.fc=nn.Sequential(nn.Linear(m.fc.in_features,256),
                        nn.ReLU(),nn.Dropout(0.3),nn.Linear(256,1))
    return m

def load_classifier(fold_idx):
    ckpt = os.path.join(CLF_ROOT, f'clf_fold_{fold_idx}.pt')
    assert os.path.exists(ckpt), f'Classifier not found: {ckpt}'
    m = build_classifier()
    m.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    return m.to(DEVICE).eval()

@torch.no_grad()
def clf_score(model, crops):
    if not crops: return []
    t=torch.stack([EVAL_TF(Image.fromarray(c)) for c in crops]).to(DEVICE)
    return torch.sigmoid(model(t).squeeze(1)).cpu().tolist()

print('Classifier loader OK')


Classifier loader OK


## [5] Two-Stage Detection

Stage 1: YOLOv11 at conf=0.05 — generates candidate footprint regions.  
Stage 2: Per-fold patch classifier — re-scores each candidate as P(missing).  
Both models use the same fold's training boards.


In [7]:
@torch.no_grad()
def yolo_proposals(yolo_model, img_path):
    res=yolo_model.predict(img_path,conf=STAGE1_CONF,iou=STAGE1_IOU,
                            imgsz=STAGE1_IMGSZ,verbose=False)[0]
    if res.boxes is None or len(res.boxes)==0: return []
    boxes=res.boxes.xyxy.cpu().numpy()
    confs=res.boxes.conf.cpu().numpy()
    return [(int(x1),int(y1),int(x2),int(y2),float(c))
            for (x1,y1,x2,y2),c in zip(boxes,confs)]

def two_stage_detect(yolo_model, clf_model, img_path, clf_thr=CLF_THR):
    img=np.array(Image.open(img_path).convert('RGB'))
    H,W=img.shape[:2]
    proposals=yolo_proposals(yolo_model, img_path)
    if not proposals: return [], 0
    crops,meta=[],[]
    for (x1,y1,x2,y2,_) in proposals:
        x1e,y1e,x2e,y2e=expand_bbox(x1,y1,x2,y2,CONTEXT_FACTOR,W,H)
        c=img[y1e:y2e,x1e:x2e]
        if c.size==0: continue
        crops.append(c); meta.append((x1,y1,x2,y2))
    if not crops: return [], len(proposals)
    scores=clf_score(clf_model, crops)
    boxes=[(x1,y1,x2,y2,sc) for (x1,y1,x2,y2),sc in zip(meta,scores)
            if sc>=clf_thr]
    return nms(boxes, NMS_IOU), len(proposals)

print('Pipeline defined')


Pipeline defined


## [6] 5-Fold Cross-Validated Evaluation

For each fold:
- Load YOLOv11 `full_dataset/fold_i` checkpoint
- Load patch classifier `clf_fold_i.pt` (trained on same fold's training boards)
- Evaluate on `missing_only/kfold_data/fold_i/valid/images`

This is fully consistent with Table III's protocol.


In [8]:
def eval_fold(fold_idx, clf_thr=CLF_THR, verbose=True):
    # Load both fold-specific models
    yolo_ckpt = os.path.join(YOLO_ROOT,'full_dataset',
                              f'fold_{fold_idx}','weights','best.pt')
    yolo = YOLO(yolo_ckpt)
    clf  = load_classifier(fold_idx)

    val_imgs = fold_val_imgs('missing_only', fold_idx)
    total_gt=tp=fp=0; scored_tp=[]; total_proposals=0

    for ip in tqdm(val_imgs, desc=f'Fold {fold_idx}', leave=False):
        labs=read_yolo_labels(label_path(ip))
        img=np.array(Image.open(ip).convert('RGB'))
        H,W=img.shape[:2]
        gt=[yolo_to_xyxy(xc,yc,w,h,W,H) for (_,xc,yc,w,h) in labs]
        total_gt+=len(gt)

        preds,n_props=two_stage_detect(yolo,clf,ip,clf_thr)
        total_proposals+=n_props

        gt_used=[False]*len(gt)
        for (x1,y1,x2,y2,sc) in preds:
            bi,biou=-1,0.
            for i,g in enumerate(gt):
                if not gt_used[i]:
                    v=iou_xyxy((x1,y1,x2,y2),g)
                    if v>biou: bi,biou=i,v
            if bi>=0 and biou>=0.5:
                gt_used[bi]=True; tp+=1; scored_tp.append((sc,1))
            else:
                fp+=1; scored_tp.append((sc,0))

    del yolo, clf; torch.cuda.empty_cache()

    prec=tp/max(1,tp+fp); rec=tp/max(1,total_gt)
    f1=2*prec*rec/max(1e-9,prec+rec); fnr=1-rec
    ap50=compute_ap_11pt(scored_tp,total_gt)
    r=dict(fold=fold_idx,total_gt=total_gt,tp=tp,fp=fp,
           total_proposals=total_proposals,
           precision=prec,recall=rec,f1=f1,fnr=fnr,ap50=ap50)
    if verbose:
        print(f'Fold {fold_idx}: GT={total_gt} TP={tp} FP={fp} '
              f'mAP={ap50:.4f} F1={f1:.4f} FNR={fnr:.4f}')
    return r

print('Running 5-fold evaluation...')
fold_results=[]
for fi in range(FOLDS):
    fold_results.append(eval_fold(fi))

ap_v  =[r['ap50'] for r in fold_results]
f1_v  =[r['f1']   for r in fold_results]
fnr_v =[r['fnr']  for r in fold_results]
prec_v=[r['precision'] for r in fold_results]

print('='*60)
print(f'MEAN mAP={np.mean(ap_v):.4f}+/-{np.std(ap_v):.4f} '
      f'F1={np.mean(f1_v):.4f}+/-{np.std(f1_v):.4f} '
      f'FNR={np.mean(fnr_v):.4f}+/-{np.std(fnr_v):.4f}')

summary=dict(fold_results=fold_results,
             summary=dict(mean_ap=np.mean(ap_v),std_ap=np.std(ap_v),
                          mean_f1=np.mean(f1_v),std_f1=np.std(f1_v),
                          mean_fnr=np.mean(fnr_v),std_fnr=np.std(fnr_v),
                          mean_prec=np.mean(prec_v),std_prec=np.std(prec_v)))
with open(os.path.join(OUT_DIR,'results_v2.json'),'w') as f:
    json.dump(summary,f,indent=2)
print('Saved ->', OUT_DIR)


Running 5-fold evaluation...


AssertionError: Classifier not found: /content/drive/MyDrive/PCB_MC/Results/pcbmc_patch_clf_v3/checkpoints/clf_fold_0.pt

## [7] Comparison Table vs All Baselines

In [ ]:
baselines=[
    ('YOLOv8  (0.25)',  0.09, 0.15, 0.90),
    ('YOLOv11 (0.25)',  0.08, 0.15, 0.87),
    ('YOLOv26 (0.25)',  0.09, 0.15, 0.87),
    ('RT-DETR (0.25)',  0.04, 0.11, 0.92),
    ('D-FINE  (0.25)',  0.03, 0.04, 0.93),
    ('Two-Stage v1 (buggy clf)', 0.362, 0.226, 0.408),
]
print(f'{"Method":<35} {"mAP@0.5":>9} {"F1":>8} {"FNR":>8}')
print('-'*62)
for name,ap,f1,fnr in baselines:
    print(f'  {name:<33} {ap:>9.4f} {f1:>8.4f} {fnr:>8.4f}')
print('-'*62)
print(f'  {"Two-Stage v2 (fold-aware)":<33} '
      f'{np.mean(ap_v):>9.4f} {np.mean(f1_v):>8.4f} {np.mean(fnr_v):>8.4f}')
print('-'*62)
print(f'\nDELTA FNR vs best YOLO: {np.mean(fnr_v)-0.87:+.4f}')
print(f'DELTA mAP vs best YOLO: {np.mean(ap_v)-0.09:+.4f}')
print(f'\nDELTA vs v1 (buggy):   FNR {np.mean(fnr_v)-0.408:+.4f}  '
      f'mAP {np.mean(ap_v)-0.362:+.4f}')


## [8] Qualitative Comparison

In [ ]:
def visualise(img_path, fold_idx, clf_thr=CLF_THR):
    yolo_ckpt=os.path.join(YOLO_ROOT,'full_dataset',
                            f'fold_{fold_idx}','weights','best.pt')
    yolo=YOLO(yolo_ckpt); clf=load_classifier(fold_idx)

    img=np.array(Image.open(img_path).convert('RGB'))
    H,W=img.shape[:2]
    labs=read_yolo_labels(label_path(img_path))
    gt=[yolo_to_xyxy(xc,yc,w,h,W,H) for (_,xc,yc,w,h) in labs]

    # Baseline: YOLO at 0.25
    yolo_std=yolo_proposals(yolo,img_path)
    # override conf for standard eval
    res_std=yolo.predict(img_path,conf=0.25,imgsz=STAGE1_IMGSZ,verbose=False)[0]
    yolo_boxes=[]
    if res_std.boxes is not None:
        for (x1,y1,x2,y2),c,cl in zip(res_std.boxes.xyxy.cpu().numpy(),
                                        res_std.boxes.conf.cpu().numpy(),
                                        res_std.boxes.cls.cpu().numpy()):
            if 'missing' in res_std.names[int(cl)].lower():
                yolo_boxes.append((int(x1),int(y1),int(x2),int(y2),float(c)))

    ts_preds,_=two_stage_detect(yolo,clf,img_path,clf_thr)
    del yolo,clf; torch.cuda.empty_cache()

    fig,axes=plt.subplots(1,3,figsize=(18,5))
    for ax,(title,color,boxes) in zip(axes,[
        ('GT Missing',(0,200,0),[(b+(None,)) for b in gt]),
        (f'YOLO conf=0.25',(220,100,0),yolo_boxes),
        (f'Two-Stage (ours)',(50,50,220),ts_preds)]):
        vis=img.copy()
        for *b,sc in boxes:
            x1,y1,x2,y2=int(b[0]),int(b[1]),int(b[2]),int(b[3])
            cv2.rectangle(vis,(x1,y1),(x2,y2),color,2)
            if sc: cv2.putText(vis,f'{sc:.2f}',(x1,max(0,y1-4)),
                               cv2.FONT_HERSHEY_SIMPLEX,0.4,color,1)
        ax.imshow(vis); ax.set_title(f'{title} ({len(boxes)})'); ax.axis('off')
    plt.suptitle(os.path.basename(img_path))
    plt.tight_layout(); plt.show()

# Show 3 examples from fold 0 validation set
val0 = fold_val_imgs('missing_only', 0)
for p in random.sample(val0, min(3, len(val0))):
    visualise(p, fold_idx=0)
